# nanochat pretraining on Kaggle

This notebook runs the **pretraining** part of the `nanochat` pipeline on Kaggle.

## Before you run it

Make sure the Kaggle notebook is configured with:

- **GPU enabled** (`2x Tesla T4` expected)
- **Internet enabled**
- Your uploaded Kaggle dataset **`nanochat-codes`** attached

The notebook auto-detects the attached code dataset from this mounted path pattern:
`/kaggle/input/datasets/<your-kaggle-username>/nanochat-codes`.

That mounted dataset is expected to contain the same project files that live under
`kaggle_dataset/nanochat/` in this repository.

## What this notebook does

The notebook runs these stages:

1. Copy the attached `nanochat-codes` dataset into `/kaggle/working/nanochat`
2. Configure caches under `/kaggle/working/nanochat_cache` and `/kaggle/working/huggingface_cache`
3. Install Python dependencies needed by the Kaggle runtime
4. Download a small ClimbMix pretraining slice
5. Train and evaluate the tokenizer
6. Pretrain a `d8` base model and save `base_checkpoints/d8_kaggle`
7. Evaluate the `d8` base model on reduced Kaggle-friendly settings
8. Optionally serve the base checkpoint or publish the cache as a Kaggle Dataset


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

DATASETS_ROOT = Path('/kaggle/input/datasets')
dataset_candidates = sorted(DATASETS_ROOT.glob('*/nanochat-codes'))

# On Kaggle, this notebook expects an attached dataset named 'nanochat-codes'.
# In this repo, the contents of that mounted dataset folder correspond to the
# local folder 'kaggle_dataset/nanochat/'. The outer 'kaggle_dataset/'
# directory is only the packaging wrapper used with the Kaggle CLI.
assert dataset_candidates, (
    "Could not find an attached Kaggle dataset matching "
    "'/kaggle/input/datasets/<username>/nanochat-codes'"
)
assert len(dataset_candidates) == 1, (
    f"Expected exactly one attached 'nanochat-codes' dataset, found: {dataset_candidates}"
)
CODE_INPUT = dataset_candidates[0]

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Code input:', CODE_INPUT)
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


In [ ]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['PYTHONFAULTHANDLER'] = '1'

print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))


## Optional Hugging Face Token

The default pretraining data is public, so this cell normally does not need a token.
Leave `USE_HF_TOKEN = False` for a normal run. Set it to `True` only if you need
authenticated Hugging Face access in your Kaggle session.


In [ ]:
USE_HF_TOKEN = False

if USE_HF_TOKEN and not os.environ.get('HF_TOKEN'):
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

if os.environ.get('HF_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN is set.')
else:
    print('Skipping HF token. Set USE_HF_TOKEN = True if you need one.')


## Install Dependencies

Install the Python packages that are needed in a fresh Kaggle runtime. Kaggle may
already provide some of them, but keeping this cell here makes the notebook easier
to rerun on a new image.


In [ ]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'pyarrow>=19.0.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


In [ ]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Download a Small Pretraining Slice

This downloads 20 train shards plus the fixed validation shard into
`/kaggle/working/nanochat_cache/base_data_climbmix`. Increase `-n` later if you
want a larger pretraining run.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'nanochat.dataset',
    '-n', '20',
    '-w', '4',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 2. Train the Tokenizer

Train the BPE tokenizer from the downloaded pretraining shards. The tokenizer is
saved under `/kaggle/working/nanochat_cache/tokenizer`.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_train',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 3. Evaluate the Tokenizer

Run a small tokenizer compression check and compare the trained tokenizer with
GPT-2 and GPT-4 style tokenizers on a few text snippets.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_eval',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 4. Pretrain a d8 Base Model

This starts a small `d8` base-model pretraining run and writes the checkpoint to
`/kaggle/working/nanochat_cache/base_checkpoints/d8_kaggle`.

The command keeps the Kaggle 2xT4 stability settings from the successful run:

- `NANOCHAT_DTYPE=float16` for Tesla T4
- `NANOCHAT_DISABLE_COMPILE=1` for Kaggle notebook stability
- optimizer communication settings for the Kaggle-safe distributed path
- NCCL settings that avoid unsupported Kaggle interconnect paths
- periodic train-time eval and sampling disabled so pretraining focuses on checkpoint creation
- training progress printed every 50 steps to keep Kaggle output readable


In [ ]:
env = os.environ.copy()
env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['NANOCHAT_TRAIN_PRINT_EVERY'] = '50'

cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.base_train',
    '--',
    '--depth=8',
    '--model-tag=d8_kaggle',
    '--run=dummy',
    '--max-seq-len=1024',
    '--window-pattern=L',
    '--device-batch-size=8',
    '--total-batch-size=262144',
    '--target-param-data-ratio=12',
    '--eval-every=-1',
    '--core-metric-every=-1',
    '--sample-every=-1',
    '--save-every=-1',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 5. Evaluate the d8 Base Model

This loads `base_checkpoints/d8_kaggle` and runs a reduced 2-GPU evaluation.
The settings are intentionally small enough for Kaggle:

- run `core`, `bpb`, and `sample`
- limit CORE examples with `--max-per-task=50`
- reduce BPB tokens with `--split-tokens=262144`

These numbers are useful for a smoke test and rough sanity check. They are not
intended as final benchmark results.


In [ ]:
env = os.environ.copy()
env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'

cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.base_eval',
    '--',
    '--eval', 'core,bpb,sample',
    '--model-tag', 'd8_kaggle',
    '--max-per-task', '50',
    '--device-batch-size', '8',
    '--split-tokens', '262144',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 6. Inspect Pretraining Artifacts

Print the cache size and a shallow file listing so you can confirm that data,
tokenizer files, reports, and base checkpoints were written where expected.


In [ ]:
print('Cache size:')
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)

print()
print('Top-level cache contents:')
subprocess.run(['find', str(WORK_CACHE), '-maxdepth', '2', '-type', 'f'], check=False)


## 7. Optional: Serve the d8 Base Model

Run this only after `base_checkpoints/d8_kaggle` exists. The cell is disabled by
default; set `RUN_SERVER = True` when you want to launch the web server.

This serves the pretrained base model, not an SFT chat model. Treat it as a
loading and generation smoke test rather than a polished assistant demo.


In [ ]:
# Optional: start a NanoChat web server for the pretrained base checkpoint.
# Set RUN_SERVER = True and run this cell when you want to serve the model.
RUN_SERVER = False

if not RUN_SERVER:
    print('Skipping server start. Set RUN_SERVER = True to launch it.')
else:
    subprocess.check_call([
        sys.executable,
        '-m', 'pip', 'install', '-q',
        'fastapi>=0.115.0',
        'pydantic>=2.0.0',
        'uvicorn[standard]>=0.30.0',
    ])

    try:
        server.terminate()
        server.wait(timeout=10)
        print('Stopped old server.')
    except NameError:
        print('No existing server variable found.')
    except Exception as e:
        print('Could not stop old server cleanly:', e)
        try:
            server.kill()
            print('Killed old server.')
        except Exception:
            pass

    env = os.environ.copy()
    env['NANOCHAT_DTYPE'] = 'float16'
    env['OMP_NUM_THREADS'] = '1'
    env['NCCL_P2P_DISABLE'] = '1'
    env['NCCL_IB_DISABLE'] = '1'

    cmd = [
        sys.executable,
        '-m', 'scripts.chat_web',
        '--source', 'base',
        '--model-tag', 'd8_kaggle',
        '--num-gpus', '1',
        '--host', '0.0.0.0',
        '--port', '8000',
    ]

    print('Running:', ' '.join(cmd))
    server = subprocess.Popen(cmd, cwd=WORK_REPO, env=env)
    print('Server starting on http://127.0.0.1:8000')


## 8. Optional: Test the Server

Run this after the server has started. The cell is disabled by default; set
`TEST_SERVER = True` to send one streaming request to `http://127.0.0.1:8000`.

Because the checkpoint is only pretrained, output quality is not expected to be
chat-like. This cell only checks that the server can accept a request and stream tokens.


In [ ]:
# Optional: smoke-test the local web server.
# Set TEST_SERVER = True after the server has started.
TEST_SERVER = False

if not TEST_SERVER:
    print('Skipping server test. Set TEST_SERVER = True after starting the server.')
else:
    import json
    import time
    import requests

    BASE_URL = 'http://127.0.0.1:8000'

    for attempt in range(30):
        try:
            health = requests.get(f'{BASE_URL}/health', timeout=5)
            health.raise_for_status()
            print('Health:', health.json())
            break
        except Exception as e:
            if attempt == 29:
                raise RuntimeError('Server did not become healthy.') from e
            time.sleep(2)

    payload = {
        'messages': [
            {'role': 'user', 'content': 'Write one short sentence about machine learning.'}
        ],
        'temperature': 0.8,
        'top_k': 50,
        'max_tokens': 80,
    }

    print('Assistant:', end=' ', flush=True)
    with requests.post(f'{BASE_URL}/chat/completions', json=payload, stream=True, timeout=120) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break
            print(data.get('token', ''), end='', flush=True)
    print()


## 9. Optional: Stop the Server

Use this when you are done testing the local server. Set `STOP_SERVER = True`
and run the cell to terminate the background process.


In [ ]:
# Optional: stop the server started above.
STOP_SERVER = False

if not STOP_SERVER:
    print('Skipping server stop. Set STOP_SERVER = True to stop it.')
else:
    try:
        server.terminate()
        server.wait(timeout=10)
        print('Server stopped.')
    except NameError:
        print('No server variable found.')
    except Exception as e:
        print('Could not stop server cleanly:', e)
        try:
            server.kill()
            print('Server killed.')
        except Exception:
            pass


## 10. Optional: Save the Pretraining Cache as a Kaggle Dataset

Use this only if you want to publish `/kaggle/working/nanochat_cache` as a
reusable Kaggle Dataset for later SFT/evaluation notebooks.

The cell is safe to leave in the notebook: it skips upload until `DATASET_ID` is
filled with a value like `your-kaggle-username/nanochat-d8-pretrain-cache`.


In [ ]:
# Optional: save /kaggle/working/nanochat_cache as a Kaggle Dataset.
# Fill this in before running the cell, for example:
# DATASET_ID = 'your-kaggle-username/nanochat-d8-pretrain-cache'
import json

CACHE_DIR = Path('/kaggle/working/nanochat_cache')
DATASET_ID = ''
TITLE = 'nanochat d8 pretrain cache'
VERSION_MESSAGE = 'Save full nanochat_cache after d8 pretraining with 20 train shards'

if not DATASET_ID:
    print('Skipping upload. Set DATASET_ID in this cell and rerun it if you want to publish the cache.')
else:
    assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
    assert CACHE_DIR.exists(), f'{CACHE_DIR} does not exist'
    assert any(CACHE_DIR.iterdir()), f'{CACHE_DIR} is empty'

    metadata = {
        'title': TITLE,
        'id': DATASET_ID,
        'licenses': [
            {'name': 'CC0-1.0'}
        ]
    }

    metadata_path = CACHE_DIR / 'dataset-metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))

    print('Dataset metadata:')
    print(metadata_path.read_text())

    print()
    print('Folder size:')
    subprocess.run(['du', '-sh', str(CACHE_DIR)], check=False)

    status = subprocess.run(
        ['kaggle', 'datasets', 'status', DATASET_ID],
        text=True,
        capture_output=True,
    )

    if status.returncode == 0:
        print()
        print(f'Dataset already exists: {DATASET_ID}')
        print('Creating a new dataset version...')
        cmd = [
            'kaggle', 'datasets', 'version',
            '-p', str(CACHE_DIR),
            '-m', VERSION_MESSAGE,
            '--dir-mode', 'tar',
        ]
    else:
        print()
        print(f'Dataset does not exist yet: {DATASET_ID}')
        print('Creating a new private dataset...')
        cmd = [
            'kaggle', 'datasets', 'create',
            '-p', str(CACHE_DIR),
            '--dir-mode', 'tar',
        ]

    print()
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True)

    if result.returncode != 0:
        raise RuntimeError('Kaggle Dataset upload failed.')

    print()
    print('Done.')
    print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')
